In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import utulek

utulek.platform.import_globals_notebook(globals())

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint = "Qwen/Qwen3.5-4B"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint,
	dtype="auto",
	device_map="auto")

In [4]:
tokenizer.response_schema = {
	"x-regex":
	r"^(?:(?:<think>)?\s*(?P<thinking>.+?)\s*</think>)?\s*(?:<tool_call>(?P<tool_calls>.*?)\s*</tool_call>)?\s*(?P<content>.+?)?\s*(?:<\|im_end\|>|$)",
	"type": "object",
	"properties": {
	"role": {
	"const": "assistant"
	},
	"content": {
	"type": "string"
	},
	"thinking": {
	"type": "string"
	},
	"tool_calls": {
	"x-regex-iterator": r"^(.*)$",
	"type": "array",
	"items": {
	"type": "object",
	"properties": {
	"type": {
	"const": "function"
	},
	"function": {
	"x-parser": "json",
	"x-parser-args": {
	"allow_non_json": True
	},
	"type": "object",
	"properties": {
	"name": {
	"type": "string"
	},
	"arguments": {
	"type": "object",
	"additionalProperties": {}
	},
	},
	},
	},
	},
	},
	},
}

In [5]:
from transformers import TextIteratorStreamer
from threading import Thread


def prompt(user_prompt: str,
	history=[{
	"role":
	"system",
	"content":
	"You are a general-purpose assistant bot. Don't overthink."
	}],
	max_new_tokens: int = 2048,
	enable_thinking: bool = True):
	messages = history
	messages.append({"role": "user", "content": user_prompt})
	input_ids = tokenizer.apply_chat_template(messages,
		add_generation_prompt=True,
		return_dict=True,
		enable_thinking=enable_thinking,
		return_tensors="pt")["input_ids"].to(model.device)
	streamer = TextIteratorStreamer(tokenizer,
		skip_prompt=True,
		skip_special_tokens=True)
	generation_args = {
		"input_ids": input_ids,
		"max_new_tokens": max_new_tokens,
		"temperature": 0.6,
		"top_p": 0.95,
		"top_k": 20,
		"min_p": 0.0,
		# "presence_penalty": 0.0,
		"repetition_penalty": 1.0,
		"do_sample": True,
		"streamer": streamer,
	}
	thread = Thread(target=model.generate,
		kwargs=generation_args)
	thread.start()
	tokens = []
	for token in streamer:
		tokens.append(token)
		yield token
	thread.join()
	history.append({
		"role": "assistant",
		"content": "".join(tokens)
	})

In [6]:
history = [{
	"role":
	"system",
	"content":
	"You are a general-purpose assistant bot. Don't overthink."
}]
for token in prompt("Give me a number.",
	history=history,
	enable_thinking=False):
	print(token, end="")
pp(history)

In [7]:
for token in prompt("Multiply by 2.",
	history=history,
	enable_thinking=False):
	print(token, end="")
pp(history)

In [8]:
for token in prompt("请给我父亲写一封信，55岁，2000字",
	history=history,
	enable_thinking=True):
	print(token, end="")

In [ ]:
history = history[:-5]

In [25]:
pp(history)

In [79]:
from transformers import TextIteratorStreamer
from threading import Thread


def prompt(user_prompt: str,
	history=[{
	"role":
	"system",
	"content":
	"You are a general-purpose assistant bot. Don't overthink."
	}],
	max_new_tokens: int = 2048,
	enable_thinking: bool = True):
	messages = history
	# messages.append({"role": "user", "content": user_prompt})
	# print(tokenizer.apply_chat_template(messages,
	# 	return_dict=True,
	# 	enable_thinking=enable_thinking,
	# 	tokenize=False,
	# 	return_tensors="pt"))
	# return
	input_ids = tokenizer.apply_chat_template(messages,
		return_dict=True,
		enable_thinking=enable_thinking,
		return_tensors="pt")["input_ids"]
	input_ids = input_ids[:, :-50].to(model.device)
	streamer = TextIteratorStreamer(tokenizer,
		skip_prompt=True,
		skip_special_tokens=True)
	generation_args = {
		"input_ids": input_ids,
		"max_new_tokens": max_new_tokens,
		"temperature": 0.6,
		"top_p": 0.95,
		"top_k": 20,
		"min_p": 0.0,
		# "presence_penalty": 0.0,
		"repetition_penalty": 1.0,
		"do_sample": True,
		"streamer": streamer,
	}
	thread = Thread(target=model.generate,
		kwargs=generation_args)
	thread.start()
	tokens = []
	for token in streamer:
		tokens.append(token)
		yield token
	thread.join()
	history.append({
		"role": "assistant",
		"content": "".join(tokens)
	})

In [80]:
for token in prompt("",
	history=history,
	enable_thinking=True,
	max_new_tokens=8192):
	print(token, end="")

In [81]:
pp(history)